# Utils

> Sorting and filtering utilities for file listings.

In [ ]:
#| default_exp components.utils

In [ ]:
#| export
from typing import List

from cjm_file_discovery.core.models import FileInfo
from cjm_fasthtml_file_browser.core.config import FilterConfig

## Sorting and Filtering

In [ ]:
#| export
def sort_files(
    files: List[FileInfo],           # Files to sort
    sort_by: str = "name",           # Sort field
    descending: bool = False,        # Sort direction
    folders_first: bool = True       # Sort folders before files
) -> List[FileInfo]:  # Sorted file list
    """Sort files by the specified field."""
    def sort_key(f: FileInfo):
        # Primary: folders first (if enabled)
        folder_order = 0 if f.is_directory else 1 if folders_first else 0
        
        # Secondary: sort field
        if sort_by == "name":
            value = f.name.lower()
        elif sort_by == "size":
            value = f.size or 0
        elif sort_by == "modified":
            value = f.modified or 0
        elif sort_by == "type":
            value = f.extension.lower() if f.extension else ""
        else:
            value = f.name.lower()
        
        return (folder_order, value)
    
    return sorted(files, key=sort_key, reverse=descending)


def filter_files(
    files: List[FileInfo],           # Files to filter
    filter_config: FilterConfig      # Filter configuration
) -> List[FileInfo]:  # Filtered file list
    """Filter files based on configuration."""
    return [f for f in files if filter_config.matches(f)]

In [ ]:
# Test sorting
files = [
    FileInfo(name="zebra.txt", path="/zebra.txt", is_directory=False, size=100, extension="txt"),
    FileInfo(name="alpha", path="/alpha", is_directory=True),
    FileInfo(name="beta.py", path="/beta.py", is_directory=False, size=50, extension="py"),
]

# Sort by name, folders first
sorted_files = sort_files(files, sort_by="name", folders_first=True)
assert sorted_files[0].name == "alpha"  # Folder first
assert sorted_files[1].name == "beta.py"  # Then files alphabetically

# Sort by size
sorted_files = sort_files(files, sort_by="size", folders_first=False)
assert sorted_files[0].size == 0 or sorted_files[0].size is None  # Folder has no size

# Test filtering
filter_cfg = FilterConfig(allowed_extensions=[".py"])
filtered = filter_files(files, filter_cfg)
assert len(filtered) == 2  # One .py file + folder (folders pass extension filter)
assert any(f.name == "beta.py" for f in filtered)
assert any(f.name == "alpha" for f in filtered)

print("Sorting and filtering tests passed!")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()